# BM25, 키워드 검색의 고전

- 벡터 검색은 "의미" 를 잡는 데 강하지만, **고유명사·코드·정확한 단어 매칭** 에는 약합니다. 
- "GPT-5.4-mini" 같은 이름을 정확히 찾으려면 키워드 검색이 더 잘 합니다.

## BM25 가 하는 일

- 문서 ↔ 쿼리의 단어 빈도 (TF) 와 희귀도 (IDF) 로 점수
- 의미를 모름, 단어가 같아야 점수가 올라감
- 임베딩 비용 0, 매우 빠름

## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [1]:
# 필요한 라이브러리 설치
# uv add -qU rank-bm25 langchain-classic langchain-cohere kiwipiepy langchain-chroma langchain-openai python-dotenv

## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
OPENAI_API_KEY=sk-...

```

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 1. BM25Retriever 한 줄 사용

In [3]:
from langchain_core.documents import Document

docs = [
    Document(page_content="마법사는 마나를 다루는 원거리 딜러.", metadata={"section": "직업"}),
    Document(page_content="GPT-5.4-mini 모델은 빠르고 저렴하다.", metadata={"section": "모델"}),
    Document(page_content="용의 둥지는 레벨 25 이상 추천.", metadata={"section": "던전"}),
    Document(page_content="기사는 검과 방패를 든 탱커.", metadata={"section": "직업"}),
    Document(page_content="claude-opus-4-7 은 Anthropic 의 최신 추론 모델.", metadata={"section": "모델"}),
]

In [4]:
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(docs, k=3) # TOP-3 반환

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_24328\2943473091.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


## 2. "GPT-5.4-mini" 같은 고유명사 검색

In [5]:
query = 'GPT-5.4-mini 가 뭐야?'
results = bm25.invoke(query)

for d in results:
    print(f" [{d.metadata['section']}] {d.page_content}")

 [모델] GPT-5.4-mini 모델은 빠르고 저렴하다.
 [모델] claude-opus-4-7 은 Anthropic 의 최신 추론 모델.
 [직업] 기사는 검과 방패를 든 탱커.


## 3. 의미는 같지만 단어가 다르면?, BM25 약점

"위자드" 와 "마법사" 는 같은 뜻이지만 BM25 는 모릅니다.

In [6]:
results = bm25.invoke("위자드 직업?")

print("=== BM25 결과 ===")
for d in results:
    print(f"  [{d.metadata['section']}] {d.page_content}")

=== BM25 결과 ===
  [모델] claude-opus-4-7 은 Anthropic 의 최신 추론 모델.
  [직업] 기사는 검과 방패를 든 탱커.
  [던전] 용의 둥지는 레벨 25 이상 추천.


> 위 결과에서 마법사 청크가 안 나오거나 점수가 낮음. BM25 는 단어가 같아야 점수가 올라가는 구조라서.

## 4. 벡터 검색과 비교

In [7]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = Chroma.from_documents(
    documents = docs,
    embedding = embeddings,
    collection_name='chroma'
)

vector_retriever = vector_store.as_retriever(search_kwargs={'k':3})


In [8]:
print("=== 벡터 검색 (의미 기반) ===")
for d in vector_retriever.invoke('전사 직업?'):
    print(f"  [{d.metadata['section']}] {d.page_content}")

print("\n=== BM25 (키워드) ===")
for d in bm25.invoke('전사 직업?'):
    print(f"  [{d.metadata['section']}] {d.page_content}")

=== 벡터 검색 (의미 기반) ===
  [직업] 기사는 검과 방패를 든 탱커.
  [던전] 용의 둥지는 레벨 25 이상 추천.
  [모델] claude-opus-4-7 은 Anthropic 의 최신 추론 모델.

=== BM25 (키워드) ===
  [모델] claude-opus-4-7 은 Anthropic 의 최신 추론 모델.
  [직업] 기사는 검과 방패를 든 탱커.
  [던전] 용의 둥지는 레벨 25 이상 추천.


## 5. 정리

| 검색 방식 | 잘하는 것 | 못하는 것 |
|---|---|---|
| **BM25 (키워드)** | 고유명사·코드·정확 단어 | 동의어·유의어·의미 변형 |
| **벡터 (의미)** | 동의어·맥락 | 고유명사·새 단어 |

두 가지를 합치면 어떨까?

## [실습]

1. `bm25.k` 를 1 / 5 / 10 으로 바꿔 결과 변화.
2. BM25 와 벡터 검색을 같은 질문 5개에 돌려 어떤 게 더 잘 잡는지 케이스별 정리.
3. BM25Retriever 가 한국어 토큰화를 어떻게 하는지 (default 는 공백 분리) 확인 후 한국어 토크나이저 직접 지정.

In [11]:
queries = [
    "전사 직업?",
    "마법사는 뭐하는데?",
    "GPS vs 클로드",
    "지금 클로드의 문제?",
    "25",
]

for query in queries:
    print("=" * 80)
    print(f"질문: {query}")

    for k in [1, 5, 10]:
        bm25 = BM25Retriever.from_documents(docs, k=k)
        vector_retriever = vector_store.as_retriever(search_kwargs={"k": k})

        print(f"\n--- k = {k} ---")

        print("=== 벡터 검색 (의미 기반) ===")
        for d in vector_retriever.invoke(query):
            print(f"  [{d.metadata['section']}] {d.page_content}")

        print("\n=== BM25 (키워드) ===")
        for d in bm25.invoke(query):
            print(f"  [{d.metadata['section']}] {d.page_content}")

질문: 전사 직업?

--- k = 1 ---
=== 벡터 검색 (의미 기반) ===
  [직업] 기사는 검과 방패를 든 탱커.

=== BM25 (키워드) ===
  [모델] claude-opus-4-7 은 Anthropic 의 최신 추론 모델.

--- k = 5 ---
=== 벡터 검색 (의미 기반) ===
  [직업] 기사는 검과 방패를 든 탱커.
  [던전] 용의 둥지는 레벨 25 이상 추천.
  [모델] claude-opus-4-7 은 Anthropic 의 최신 추론 모델.
  [모델] GPT-5.4-mini 모델은 빠르고 저렴하다.
  [직업] 마법사는 마나를 다루는 원거리 딜러.

=== BM25 (키워드) ===
  [모델] claude-opus-4-7 은 Anthropic 의 최신 추론 모델.
  [직업] 기사는 검과 방패를 든 탱커.
  [던전] 용의 둥지는 레벨 25 이상 추천.
  [모델] GPT-5.4-mini 모델은 빠르고 저렴하다.
  [직업] 마법사는 마나를 다루는 원거리 딜러.

--- k = 10 ---
=== 벡터 검색 (의미 기반) ===
  [직업] 기사는 검과 방패를 든 탱커.
  [던전] 용의 둥지는 레벨 25 이상 추천.
  [모델] claude-opus-4-7 은 Anthropic 의 최신 추론 모델.
  [모델] GPT-5.4-mini 모델은 빠르고 저렴하다.
  [직업] 마법사는 마나를 다루는 원거리 딜러.

=== BM25 (키워드) ===
  [모델] claude-opus-4-7 은 Anthropic 의 최신 추론 모델.
  [직업] 기사는 검과 방패를 든 탱커.
  [던전] 용의 둥지는 레벨 25 이상 추천.
  [모델] GPT-5.4-mini 모델은 빠르고 저렴하다.
  [직업] 마법사는 마나를 다루는 원거리 딜러.
질문: 마법사는 뭐하는데?

--- k = 1 ---
=== 벡터 검색 (의미 기반) ===
  [직업] 마법사는 마나를 다루는 원거리 딜러.

=== BM25 (키워드) ===
  [직업]